# clue_report

> Phase 11 report: the `CLUE_PLAN` analyses (main table with team totals, channel benefit, information use, numpy vs LLM, framing, liar, optional arms), `report.md` with provenance first, and four figures. All findings are exploratory.

In [ ]:
#| default_exp clue_report

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

from explRL.llm_compare import compareNumpy
from explRL.llm_figures import COLORS, _legend, _save
from explRL.llm_report import NAN, _f, _fmt, _paired, _sel, _stat, interfaceTable

CH = tuple(COLORS)
INFO = ('sharePerHeld', 'redundantShares', 'assisted', 'shareToUse', 'falseShares')


def _cells(rows, metrics):
    out = {}
    for r in rows:
        out.setdefault((r['channel'], int(_f(r['N']))), []).append(r)
    return {k: {m: _stat(_f(r.get(m, NAN)) for r in v) for m in metrics} for k, v in sorted(out.items())}


def summaries(rows, numpyRows=()):
    "The CLUE_PLAN analyses: main table, channel benefit, information, numpy vs LLM, framing, liar, optional arms."
    main = _sel(rows, study='clue', runner='llm')
    S = dict(main=_cells(main, ('welfare', 'teamWelfare', 'pooledShare', 'invalid', 'calls') + INFO))
    S['benefit'] = {(ch, N): _stat(_paired(_sel(main, channel=ch, N=N), _sel(main, channel='none', N=N), 'welfare'))
                    for ch, N in S['main'] if ch != 'none'}
    S['numpy'] = compareNumpy(main, list(numpyRows)).get('clue') if numpyRows else None
    S['numpyCells'] = _cells(list(numpyRows), ('welfare', 'sharePerHeld')) if numpyRows else {}
    fr = _sel(rows, study='clueFraming')
    S['framing'] = {f'team - own, N = {N}: {m}': _stat(_paired(_sel(fr, level='framing=team', N=N),
                                                               _sel(fr, level='framing=own', N=N), m))
                    for N in (4, 8) for m in ('sharePerHeld', 'welfare')}
    clean = [r for r in main if int(_f(r['N'])) == 4 and int(_f(r['seed'])) in (900, 901, 902)]
    li, loss = _sel(rows, study='clueLiar'), {}
    for ch in ('blackboard', 'messages'):
        c = {int(_f(r['seed'])): _f(r['welfareHonest']) for r in clean if r['channel'] == ch}
        lr = {int(_f(r['seed'])): _f(r['welfareHonest']) for r in li if r['channel'] == ch}
        loss[ch] = {s: c[s] - lr[s] for s in sorted(set(c) & set(lr))}
    S['liar'] = {f'honest loss, {ch}': _stat(v.values()) for ch, v in loss.items()}
    both = sorted(set(loss['blackboard']) & set(loss['messages']))
    S['liar']['blackboard - messages loss (H3 sign: > 0)'] = _stat(loss['blackboard'][s] - loss['messages'][s] for s in both)
    ef = _sel(rows, study='clueEffort')
    ben = lambda lv: dict(zip([int(_f(r['seed'])) for r in _sel(ef, level=lv, channel='blackboard')],
                              _paired(_sel(ef, level=lv, channel='blackboard'), _sel(ef, level=lv, channel='none'), 'welfare')))
    s, h = ben('sonnet|medium'), ben('haiku|medium')
    S['optional'] = {'Sonnet - Haiku blackboard benefit, N = 4, effort medium': _stat(s[k] - h[k] for k in sorted(set(s) & set(h)))}
    S['optional'].update({f'scale N = 16, {ch}: welfare': _stat(_f(r['welfare']) for r in _sel(rows, study='clueScale', channel=ch))
                          for ch in ('none', 'blackboard')})
    S['interface'] = interfaceTable(rows)
    return S


def writeReport(path, S, meta):
    "report.md: provenance first, then the analysis-plan tables (all exploratory)."
    L = [f"# Phase 11: the clue board ({meta['mode']})", '', 'Exploratory, pre-specified in `CLUE_PLAN`.', '',
         f"- Plan hash `{meta['planHash']}`, git commit `{meta['commit']}`; settings {meta['settings']}",
         f"- Models {', '.join(meta['models']) or 'none'}; prices checked {meta['pricesChecked']}; cost of these runs "
         f"${meta['cost']:.2f}; total spend ${meta['spent']:.2f} of the ${meta['cap']:.0f} cap; calls {meta['calls']}; "
         f"wall time {meta['wall']:.0f} s; retries {meta.get('retries', 0)}", '',
         '## Main grid (welfare per agent-turn, team total, pooled-bound share)', '',
         '| channel | N | welfare | team total | pooled share | shares per held clue | assisted | share-to-use (turns) |',
         '|---|---|---|---|---|---|---|---|']
    L += [f"| {ch} | {N} | {_fmt(v['welfare'])} | {_fmt(v['teamWelfare'])} | {v['pooledShare']['mean']:.2f} | "
          f"{v['sharePerHeld']['mean']:.2f} | {v['assisted']['mean']:.2f} | {v['shareToUse']['mean']:.1f} |"
          for (ch, N), v in S['main'].items()]
    L += ['', '## Channel benefit (minus none, paired on seed)', '', '| channel | N | benefit |', '|---|---|---|']
    L += [f'| {ch} | {N} | {_fmt(v)} |' for (ch, N), v in S['benefit'].items()]
    if S['numpy']:
        v = S['numpy']
        L += ['', f"## Numpy predicts LLM? Spearman rho {v['rho']:.2f} [{v['ci'][0]:.2f}, {v['ci'][1]:.2f}] over "
                  f"{v['cells']} cells, {v['seeds']} seeds"]
    for title, key in (('Framing (team minus own)', 'framing'), ('Liar (one false-clue liar among 4)', 'liar'),
                       ('Optional arms', 'optional')):
        L += ['', f'## {title}', '', '| contrast | mean [95% CI] | n |', '|---|---|---|']
        L += [f"| {k} | {_fmt(v)} | {v['n']} |" for k, v in S[key].items()]
    L += ['', '## Interface, tokens and cost', '', '| study | runs | calls | invalid | refusals | cost $ |', '|---|---|---|---|---|---|']
    L += [f"| {st} | {v['runs']} | {v['calls']} | {v['invalid']:.3f} | {v['refusals']} | {v['cost']:.2f} |"
          for st, v in S['interface'].items()]
    path.write_text('\n'.join(L) + '\n')
    return path


def _lines(ax, cells, metric, ylabel, title):
    for ch in CH:
        pts = sorted((N, v[metric]) for (c, N), v in cells.items() if c == ch)
        if pts:
            m = np.array([p[1]['mean'] for p in pts])
            err = [m - np.array([p[1]['ci'][0] for p in pts]), np.array([p[1]['ci'][1] for p in pts]) - m]
            ax.errorbar([p[0] for p in pts], m, yerr=err, color=COLORS[ch], marker='o', capsize=3, label=ch)
    ax.set(xscale='log', xlabel='team size N', ylabel=ylabel, title=title)
    ax.set_xticks([2, 4, 8], ['2', '4', '8'])
    ax.minorticks_off()


def writeFigures(out, S):
    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    _lines(axs[0], S['main'], 'welfare', 'welfare per agent-turn', 'Clue board: welfare')
    _lines(axs[1], S['main'], 'teamWelfare', 'team total', 'Clue board: team total')
    _legend(axs[0])
    paths = [_save(fig, out / 'clue_welfare.png')]
    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    for ax, (m, lab) in zip(axs, (('sharePerHeld', 'shares per held clue'), ('assisted', 'share of solves assisted'),
                                  ('shareToUse', 'share-to-use time (turns)'))):
        _lines(ax, S['main'], m, lab, lab)
    _legend(axs[0])
    paths.append(_save(fig, out / 'clue_info.png'))
    fig, ax = plt.subplots(figsize=(5, 4))
    for (ch, N), v in S['main'].items():
        if (ch, N) in S['numpyCells']:
            ax.scatter(S['numpyCells'][(ch, N)]['welfare']['mean'], v['welfare']['mean'], color=COLORS[ch], s=10 + 8 * N,
                       label=ch if N == 2 else None)
    rho = f" (rho {S['numpy']['rho']:.2f})" if S['numpy'] else ''
    ax.set(xlabel='rule agents: welfare per agent-turn', ylabel='LLM agents: welfare per agent-turn', title='Numpy vs LLM' + rho)
    _legend(ax)
    paths.append(_save(fig, out / 'clue_numpy.png'))
    items = list(S['framing'].items()) + list(S['liar'].items())
    fig, ax = plt.subplots(figsize=(9, 0.35 * len(items) + 1.2))
    m = np.array([v['mean'] for _, v in items])
    err = [m - np.array([v['ci'][0] for _, v in items]), np.array([v['ci'][1] for _, v in items]) - m]
    ax.errorbar(m, range(len(items)), xerr=err, fmt='o', capsize=3)
    ax.set_yticks(range(len(items)), [k for k, _ in items], fontsize=7)
    ax.axvline(0, color='k', lw=0.5)
    ax.invert_yaxis()
    ax.set(xlabel='difference', title='Framing and liar')
    paths.append(_save(fig, out / 'clue_arms.png'))
    return paths

## Tests

In [ ]:
import math
import tempfile
from pathlib import Path

from explRL.clue_grid import clueHash, clueQuick
from explRL.clue_runs import runClueAny
from explRL.llm_sched import runWorkload

# Synthetic rows with known differences: the paired benefit and the liar losses come out exactly.
def row(study, ch, N, seed, w, level='l', **kw):
    return dict(study=study, runner='llm', task='clue', channel=ch, N=N, seed=seed, level=level, welfare=w,
                welfareHonest=w, teamWelfare=w * N, pooledShare=0.5, invalid=0, calls=10, refusals=0, maxTokenStops=0,
                promptCacheRead=0.9, inputTokens=1, cacheWriteTokens=0, cacheReadTokens=0, outputTokens=1, cost=0.01,
                sharePerHeld=0.3, redundantShares=0, assisted=0.5, shareToUse=2.0, falseShares=0, **kw)
rows = [row('clue', ch, N, s, base + s - 900) for ch, base in (('none', 1.0), ('blackboard', 1.5), ('messages', 1.2))
        for N in (2, 4, 8) for s in range(900, 905)]
rows += [row('clueLiar', ch, 4, s, base + s - 900) for ch, base in (('blackboard', 1.0), ('messages', 1.1)) for s in (900, 901, 902)]
S = summaries(rows)
assert abs(S['benefit'][('blackboard', 4)]['mean'] - 0.5) < 1e-12 and S['benefit'][('blackboard', 4)]['n'] == 5
assert abs(S['liar']['honest loss, blackboard']['mean'] - 0.5) < 1e-12
assert abs(S['liar']['honest loss, messages']['mean'] - 0.1) < 1e-12
assert abs(S['liar']['blackboard - messages loss (H3 sign: > 0)']['mean'] - 0.4) < 1e-12
assert math.isnan(S['framing']['team - own, N = 4: welfare']['mean']) and S['numpy'] is None

# The quick fake-client workload: report and all four figures, with provenance first.
qrows = runWorkload([('quick', clueQuick())], dict(kind='fake'), 1, run=runClueAny)['rows']
llm, npr = [r for r in qrows if r['runner'] == 'llm'], [r for r in qrows if r['runner'] == 'numpy' and r['study'] == 'clueNumpy']
S = summaries(llm, npr)
with tempfile.TemporaryDirectory() as d:
    meta = dict(mode='quick', planHash=clueHash(), commit='abc', settings='m=3', models=['claude-haiku-5-5'], cap=150,
                wall=1.0, pricesChecked='2026-10-12', cost=0.0, calls=1, spent=0.0)
    text = writeReport(Path(d) / 'report.md', S, meta).read_text()
    assert text.index(clueHash()) < text.index('## Main grid') and 'claude-haiku-5-5' in text and '$150 cap' in text
    paths = writeFigures(Path(d), S)
    assert [p.name for p in paths] == ['clue_welfare.png', 'clue_info.png', 'clue_numpy.png', 'clue_arms.png']
    assert all(p.stat().st_size > 0 for p in paths)
print(text[:600])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()